In [5]:
# Packages used across the notebook
import os
import re
import math
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Rectangle, FancyBboxPatch
from matplotlib.ticker import MultipleLocator

from scipy import stats
from scipy.spatial.distance import pdist, squareform

from sklearn.preprocessing import StandardScaler

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"

In [ ]:
# ── Figure setup ──
fig, ax = plt.subplots(figsize=(7.5, 6), dpi=300)

# ── Create CI heatmap surface ──
x = np.linspace(0, 1, 500)  
y = np.linspace(0, 1, 500)  
X, Y = np.meshgrid(x, y)
CI = X * Y

# Custom colormap: red → white → blue
colors_list = [
    (0.00, '#E28187'),
    (0.25, '#EBBFC2'),
    (0.50, '#D5E1E3'),
    (0.75, '#AFC9CF'),
    (1.00, '#8FB4BE'),
]
cmap = mcolors.LinearSegmentedColormap.from_list(
    'ci_cmap', [(v, c) for v, c in colors_list], N=256
)

# Plot heatmap
im = ax.imshow(
    CI, extent=[0, 1, 0, 1], origin='lower',
    cmap=cmap, vmin=0, vmax=1, aspect='auto'
)

# ── CI isolines ──
contour_levels = [0.01, 0.04, 0.08, 0.10, 0.12, 0.16, 0.20, 0.24, 0.30,
                  0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
CS = ax.contour(
    X, Y, CI, levels=contour_levels,
    colors='#2C2C2A', linewidths=0.5, alpha=0.30
)
ax.clabel(CS, inline=True, fontsize=14, fmt='%.2f', colors='#333333')

# ── Example points ──
examples = [
    {'avgR': 0.67, 'vpap': 0.60, 'label': 'A',
     'desc': 'High Avg|R|\nhigh VP/TP',
     'offset': (20, 15), 'ha': 'left'},
    {'avgR': 0.50, 'vpap': 0.20, 'label': 'B',
     'desc': 'Moderate Avg|R|\nlow VP/TP',
     'offset': (20, -15), 'ha': 'left'},
    {'avgR': 0.20, 'vpap': 0.50, 'label': 'C',
     'desc': 'Low Avg|R|\nmoderate VP/TP',
     'offset': (20, 15), 'ha': 'left'},
    {'avgR': 0.10, 'vpap': 0.10, 'label': 'D',
     'desc': 'Low Avg|R|\nlow VP/TP',
     'offset': (20, 12), 'ha': 'left'},
]

for e in examples:
    e['ci'] = e['avgR'] * e['vpap']

for e in examples:
    ax.plot(e['avgR'], e['vpap'], 'o',
            markersize=10, markerfacecolor='white',
            markeredgecolor='#0b0b0b', markeredgewidth=1.8, zorder=5)
    ax.plot(e['avgR'], e['vpap'], 'o',
            markersize=3, markerfacecolor='#0b0b0b',
            markeredgecolor='#0b0b0b', zorder=6)

    ax.annotate(
        f'{e["label"]}  CI = {e["ci"]:.2f}\n{e["desc"]}',
        xy=(e['avgR'], e['vpap']),
        xytext=e['offset'],
        textcoords='offset points',
        fontsize=12,
        color='#2C2C2A',
        ha=e['ha'], va='center',
        bbox=dict(
            boxstyle='round,pad=0.3',
            facecolor='white', edgecolor='#AEB6BF',
            alpha=0.88, linewidth=0.5
        ),
        arrowprops=dict(
            arrowstyle='-', color='#5D6D7E', linewidth=0.8
        ),
        zorder=7
    )

# ── B-C connection along CI = 0.10 isoline ──
t = np.linspace(0.20, 0.50, 200)
ci_bc = 0.10
curve_y = ci_bc / t
mask = (curve_y >= 0) & (curve_y <= 1)
ax.plot(t[mask], curve_y[mask], '--', color='#922B21', linewidth=1.5, alpha=0.8, zorder=4)

ax.text(
    0.42, 0.39,
    'B and C on same\nisoline: CI = 0.10\nbut different paths',
    fontsize=12, color='#922B21',
    ha='center', va='center',
    fontstyle='italic',
    bbox=dict(
        boxstyle='round,pad=0.3',
        facecolor='white', edgecolor='#E6B0AA',
        alpha=0.88, linewidth=0.5
    ),
    zorder=7
)

# ── Axes ──
ax.set_xlabel(r'Average absolute correlation  (Avg$|\mathrm{R}|$)', fontsize=15, labelpad=8)
ax.set_ylabel('Valid pair ratio  (VP / TP)', fontsize=15, labelpad=8)

# ── Colorbar ──
cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('CI value', fontsize=15, labelpad=8)
cbar.ax.tick_params(labelsize=9)

# ── Ticks ──
ax.set_xticks(np.arange(0, 1.1, 0.2))
ax.set_yticks(np.arange(0, 1.1, 0.2))
ax.tick_params(labelsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "1_CI_heatmap.png", dpi=300, bbox_inches='tight')
print('Done.')
print('Verification:')
for e in examples:
    print(f'  {e["label"]}: {e["avgR"]:.2f} x {e["vpap"]:.2f} = {e["ci"]:.3f}')

In [ ]:
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.linewidth'] = 0.6

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

def draw_panel(ax, title, ci_syn1, ci_tra1, ci_syn2=None, ci_tra2=None,
               dom_type='', dom_color='#333333', keep_two=False):

    dom1 = ci_syn1 - ci_tra1
    dom2 = None if ci_syn2 is None or ci_tra2 is None else ci_syn2 - ci_tra2

    ax.set_xlim(0, 0.30)
    ax.set_ylim(-0.7, 4.2)
    ax.set_title(title, fontsize=16, fontweight='medium', pad=8, color=dom_color)
    ax.set_xlabel('CI value', fontsize=14)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # More compact vertical layout
    y_syn_center = 2.60
    y_tra_center = 1.10
    y_dash = 1.85

    if keep_two:
        y_syn1, y_syn2 = 2.85, 2.35
        y_tra1, y_tra2 = 1.35, 0.85
    else:
        y_syn1 = y_syn_center
        y_tra1 = y_tra_center

    ax.set_yticks([y_syn_center, y_tra_center])
    ax.set_yticklabels(
        [r'$\mathrm{CI}_{\mathrm{synergy}}$',
         r'$\mathrm{CI}_{\mathrm{trade\!-\!off}}$'],
        fontsize=14, fontweight='medium'
    )

    # Keep outward y ticks
    ax.tick_params(axis='y', direction='out', length=5, width=0.6, left=True)

    # Dashed line starts from y-axis
    ax.axhline(y_dash, color='#888888', linewidth=1, linestyle='--', xmin=0, xmax=0.95)

    bar_h = 0.32
    syn_color = '#4A90D9'
    tra_color = '#D93F49'

    def draw_bar(y, ci, label, color):
        ax.barh(y, ci, height=bar_h, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
        xpos = 0.01 if ci == 0 else min(ci + 0.008, 0.285)
        text = f'{label} = {ci:.2f}' if label else f'{ci:.2f}'
        ax.text(xpos, y, text, va='center', ha='left', fontsize=11, color='black')

    if keep_two:
        draw_bar(y_syn1, ci_syn1, '(1)', syn_color)
        draw_bar(y_syn2, ci_syn2, '(2)', syn_color)
        draw_bar(y_tra1, ci_tra1, '(1)', tra_color)
        draw_bar(y_tra2, ci_tra2, '(2)', tra_color)
        calc_line1 = f'(1)  {ci_syn1:.2f} − {ci_tra1:.2f} = {dom1:+.2f}'
        calc_line2 = f'(2)  {ci_syn2:.2f} − {ci_tra2:.2f} = {dom2:+.2f}'
        calc_text = f'{calc_line1}\n{calc_line2}'
    else:
        draw_bar(y_syn1, ci_syn1, '', syn_color)
        draw_bar(y_tra1, ci_tra1, '', tra_color)
        calc_text = f'{ci_syn1:.2f} − {ci_tra1:.2f} = {dom1:+.2f}'

    # Move the dominance-score text lower
    ax.text(0.15, 3.72, dom_type, ha='center', va='center', fontsize=12, color='black')

    # Move the calculation box lower
    ax.text(0.15, -0.18, calc_text, ha='center', va='center', fontsize=11.5,
            bbox=dict(boxstyle='round,pad=0.35', facecolor='#F8F8F8', edgecolor=dom_color, linewidth=1))

    ax.set_xticks([0, 0.1, 0.2, 0.3])
    ax.tick_params(axis='x', direction='out', length=4, width=0.6, labelsize=11)

# a. Synergy dominance
draw_panel(
    axes[0],
    'Synergy dominance',
    ci_syn1=0.20,
    ci_tra1=0.08,
    dom_type='Dominance score > 0',
    dom_color='#4A90D9',
    keep_two=False
)

# b. Trade-off dominance
draw_panel(
    axes[1],
    'Trade-off dominance',
    ci_syn1=0.05,
    ci_tra1=0.18,
    dom_type='Dominance score < 0',
    dom_color='#D93F49',
    keep_two=False
)

# c. No dominance
draw_panel(
    axes[2],
    'No dominance',
    ci_syn1=0.12,
    ci_tra1=0.12,
    ci_syn2=0.00,
    ci_tra2=0.00,
    dom_type='Dominance score = 0',
    dom_color='#888888',
    keep_two=True
)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "1_dominance_score.png",
            dpi=300, bbox_inches='tight')

print('Verification:')
print(f'  (a)  0.20 - 0.08 = {0.20 - 0.08:+.2f}')
print(f'  (b)  0.05 - 0.18 = {0.05 - 0.18:+.2f}')
print(f'  (c1) 0.12 - 0.12 = {0.12 - 0.12:+.2f}')
print(f'  (c2) 0.00 - 0.00 = {0.00 - 0.00:+.2f}')
print('Done.')

In [ ]:
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.linewidth'] = 0.6

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5.5))

def draw_network(ax, title, hub_scores, edge_data, node_color, highlight_color, annot_offset=(0.5, 0.4)):
    """Draw a simplified SDG network showing hub score concept."""
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('', pad=5)
    ax.text(0, -1.35, title, ha='center', va='top', fontsize=16, fontweight='medium')

    n = len(hub_scores)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False) - np.pi / 2
    positions = {i: (np.cos(a), np.sin(a)) for i, a in enumerate(angles)}

    # Draw edges
    for (i, j, w) in edge_data:
        x1, y1 = positions[i]
        x2, y2 = positions[j]
        ax.plot([x1, x2], [y1, y2], color=node_color, alpha=0.3, linewidth=0.5 + 3 * w)

    # Draw nodes
    sdg_labels = ['SDG1', 'SDG3', 'SDG5', 'SDG7', 'SDG9', 'SDG11', 'SDG12', 'SDG16']
    max_hub = max(hub_scores)
    for i, (label, hub) in enumerate(zip(sdg_labels, hub_scores)):
        x, y = positions[i]
        size = 300 + 700 * (hub / max_hub)
        c = node_color
        alpha = 0.4
        ax.scatter(x, y, s=size, color=c, alpha=alpha, edgecolors='black', linewidths=0.8, zorder=3)
        ax.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold', color='black', zorder=4)
        # Hub score label
        ax.text(x, y - 0.18, f'{hub:.2f}', ha='center', va='top', fontsize=11, color='black', zorder=4)

    # Legend annotation for top hub
    top_idx = hub_scores.index(max(hub_scores))
    tx, ty = positions[top_idx]
    ax.annotate(
        f'Highest hub score\n({max_hub:.2f})',
        xy=(tx, ty), xytext=(tx + annot_offset[0], ty + annot_offset[1]),
        fontsize=12, ha='center', color=highlight_color, fontweight='medium',
        arrowprops=dict(arrowstyle='<-', color=highlight_color, linewidth=1),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=highlight_color, alpha=0.9, linewidth=0.5)
    )

# Synergy network
syn_hubs = [0.65, 0.55, 0.20, 0.30, 1.00, 0.40, 0.15, 0.70]
syn_edges = [
    (0, 1, 0.8), (0, 4, 0.9), (0, 7, 0.7), (1, 4, 0.85),
    (1, 7, 0.6), (3, 4, 0.5), (4, 5, 0.7), (4, 7, 0.9),
    (0, 3, 0.4), (1, 5, 0.3), (5, 7, 0.5), (2, 5, 0.2),
    (3, 6, 0.2), (2, 6, 0.15),
]
draw_network(axes2[0], 'Synergy network', syn_hubs, syn_edges, '#4A90D9', '#2171B5', annot_offset=(0.5, 0.2))

# Tradeoff network
tra_hubs = [0.25, 0.35, 0.77, 0.45, 0.10, 1.00, 0.75, 0.20]
tra_edges = [
    (2, 5, 0.9), (5, 6, 0.85), (2, 6, 0.7), (2, 3, 0.5),
    (3, 5, 0.6), (5, 0, 0.4), (5, 1, 0.5), (6, 1, 0.4),
    (6, 3, 0.3), (0, 3, 0.2), (2, 4, 0.15), (5, 7, 0.3),
    (5, 4, 0.25), (6, 7, 0.2),
]
draw_network(axes2[1], 'Tradeoff network', tra_hubs, tra_edges, '#D93F49', '#B03A2E', annot_offset=(0.5, 0.5)) 


plt.tight_layout()
plt.savefig(RESULTS_DIR / "1_hub_score_illustration.png", dpi=300, bbox_inches='tight')
print('Hub score figure done.')

In [ ]:
"""
Linear Regression Analysis: SDG Score vs. Three Factors
Generates a 3-panel figure with scatter, linear fit, and 95% CI.
"""
# =============================================================================
# 1. Load data
# =============================================================================
PATH_DATA = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result1.xlsx')

df = pd.read_excel(PATH_DATA)
df.columns = [c.strip() for c in df.columns]
print("Columns in file:", list(df.columns))
print(df.head())

Y_COL = 'SDG Score'
X_COL_POP = 'POP density (pop/km2)'
X_COL_GDP = 'GDP per capita (thousands USD)'
X_COL_GOV = 'Governance capacity'    

y = df[Y_COL].values
n = len(y)

# =============================================================================
# 2. Plot configuration
# =============================================================================
specs = [
    (X_COL_POP, 'Population density (pop/km²)',
     '#B86B0A', '#F5DDB8'),   # (column, x-label, line_color, CI_color)
    (X_COL_GDP, 'GDP per capita (thousand USD)',
     '#6C4DA0', '#D8CCE8'),
    (X_COL_GOV, 'Governance capacity',
     '#1A8C6E', '#C2E5D8'),
]

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.4), sharey=True)
plt.rcParams['font.family'] = 'Arial'

# =============================================================================
# 3. Loop over factors and draw regression
# =============================================================================
for ax, (col, xlabel, line_color, ci_color) in zip(axes, specs):
    x = df[col].values.astype(float)

    # Ordinary Least Squares
    slope, intercept, r, p, se_slope = stats.linregress(x, y)

    # Fitted line + 95% CI for mean response
    xg = np.linspace(x.min(), x.max(), 240)
    yhat = intercept + slope * xg
    resid = y - (intercept + slope * x)
    s_err = np.sqrt(np.sum(resid**2) / (n - 2))
    xm = x.mean()
    se_mean = s_err * np.sqrt(1/n + (xg - xm)**2 / np.sum((x - xm)**2))
    tcrit = stats.t.ppf(0.975, n - 2)
    band = tcrit * se_mean

    # Scatter
    ax.scatter(x, y, s=44, color=line_color, alpha=0.55,
               edgecolor='white', linewidth=0.6, zorder=2)
    # 95% CI band
    ax.fill_between(xg, yhat - band, yhat + band,
                    color=ci_color, alpha=0.6, zorder=1, label='95% CI')
    # Fitted line
    ax.plot(xg, yhat, color=line_color, lw=2.4, zorder=3, label='Linear fit')

    # Labels and cosmetics
    short = xlabel.split(' (')[0]
    ax.set_xlabel(xlabel, fontsize=14)
    ax.tick_params(axis='both', labelsize=12)
    if ax == axes[0]:
        ax.set_ylabel('SDG Score', fontsize=14)
    ax.grid(True, alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_major_locator(MultipleLocator(10))

    # Stats box (R² and p-value)
    r2 = r ** 2
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    p_str = '< 0.001' if p < 0.001 else f'= {p:.3f}'
    txt = f"R² = {r2:.3f}\np {p_str} {sig}"
    ax.text(0.04, 0.96, txt, transform=ax.transAxes, fontsize=12,
            va='top', ha='left', family='monospace',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                      edgecolor='#cccccc', alpha=0.93))

    # Legend on every panel (colored to match)
    ax.legend(loc='lower right', frameon=False, fontsize=12)

    # Print numerical results
    print(f"\n{xlabel}:")
    print(f"  slope = {slope:.4g}")
    print(f"  intercept = {intercept:.4g}")
    print(f"  R² = {r2:.4f}")
    print(f"  p = {p:.4g}")

# =============================================================================
# 4. Save figure
# =============================================================================
plt.tight_layout()
plt.savefig(RESULTS_DIR / "2_SDG_linear_regression.png",
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
"""
SDG Indicator Centroid Heatmap by Cluster
"""
# =============================================================================
# Global font settings — Arial for all plots
# =============================================================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12

# =============================================================================
# 1. Load data
# =============================================================================
PATH_CLUSTERS = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result2_Indicator_centroids.xlsx')

# --------------------------
df = pd.read_excel(PATH_DATA)
df.columns = [c.strip() for c in df.columns]

df = df.set_index('Cluster')
centroids = df.T  # rows = SDG indicators, columns = Cluster1/2/3
centroids.index.name = 'SDG Indicator'

rename_map = {'Cluster1': 'Typology 1', 'Cluster2': 'Typology 2', 'Cluster3': 'Typology 3'}
centroids = centroids.rename(columns=rename_map)

def sort_key(s):
    m = re.match(r'G(\d+)\.(\d+)', s)
    return (int(m.group(1)), int(m.group(2))) if m else (999, 0)

centroids = centroids.loc[sorted(centroids.index, key=sort_key)]

print(f"Indicators × Clusters: {centroids.shape}")
print(centroids.head())

# =============================================================================
# 2. Colormap (red-white-blue diverging)
# =============================================================================
cmap = LinearSegmentedColormap.from_list('cluster_div',
    ['#8B1A1A', '#C87878', '#F4E4E4', '#FFFFFF', '#DCE4EE', '#8FA5C4', '#3A5578'], N=256)

vmax = max(abs(centroids.values.min()), abs(centroids.values.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# =============================================================================
# 3. Plot
# =============================================================================
fig, ax = plt.subplots(figsize=(7.2, 14))
im = ax.imshow(centroids.values, aspect='auto', cmap=cmap, norm=norm)

for i in range(centroids.shape[0]):
    for j in range(centroids.shape[1]):
        val = centroids.values[i, j]
        # White text on dark cells, black on light cells
        text_color = 'white' if abs(val) > vmax * 0.55 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color=text_color)

ax.set_yticks(np.arange(centroids.shape[0]))
ax.set_yticklabels(centroids.index, fontsize=12)

ax.set_xticks(np.arange(centroids.shape[1]))
ax.set_xticklabels(centroids.columns, fontsize=18, fontweight='normal')

prev_goal = None
for i, name in enumerate(centroids.index):
    g = int(re.match(r'G(\d+)', name).group(1))
    if prev_goal is not None and g != prev_goal:
        ax.axhline(i - 0.5, color='black', lw=0.6, alpha=0.5)
    prev_goal = g

for j in range(1, centroids.shape[1]):
    ax.axvline(j - 0.5, color='white', lw=1.2)

cbar = plt.colorbar(im, ax=ax, fraction=0.05, pad=0.03, shrink=0.5)
cbar.set_label('Centroid value\n(below ← 0 → above overall mean)', fontsize=18)
cbar.ax.tick_params(labelsize=12)

ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

plt.tight_layout()

# =============================================================================
# 4. Export
# =============================================================================
fig.savefig(os.path.join(RESULTS_DIR, 'SDG_centroids_heatmap.png'),
            dpi=200, bbox_inches='tight', facecolor='white')

plt.show()
print("\nDone. Figure saved as SDG_centroids_heatmap.png")

In [ ]:
"""
Kruskal-Wallis + Dunn's Post-hoc Boxplots for 13 Influencing Factors
"""
# =============================================================================
# Global font settings — Arial
# =============================================================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12

# =============================================================================
# 1. Load data
# =============================================================================
PATH_CLUSTERS = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result2.xlsx')
PATH_CLUSTERS = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result2_ALLfactors.xlsx')

df = pd.read_excel(PATH_FACTORS)
df.columns = [c.strip() for c in df.columns]

if 'Cluster' not in df.columns:
    df_clu = pd.read_excel(PATH_CLUSTERS)
    df_clu.columns = [c.strip() for c in df_clu.columns]
    df = df.merge(df_clu[['city', 'Cluster']], on='city')

factor_cols = [c for c in df.columns
               if c not in ['num', 'city', 'MSA', 'Cluster']]

for c in factor_cols:
    if df[c].isna().any():
        med = df[c].median()
        n_na = df[c].isna().sum()
        df[c] = df[c].fillna(med)
        print(f"  Filled {n_na} NaN in '{c}' with median={med:.4f}")

labels = df['Cluster'].values
print(f"Factors: {len(factor_cols)}")
print(f"Cluster sizes: { {c: int((labels==c).sum()) for c in sorted(set(labels))} }")

# =============================================================================
# 2. Statistical tests
# =============================================================================
def dunn_test(groups):
    """Dunn's post-hoc test with Bonferroni correction."""
    all_vals = np.concatenate(list(groups.values()))
    ranks = stats.rankdata(all_vals)
    rank_groups = {}
    idx = 0
    for k, v in groups.items():
        rank_groups[k] = ranks[idx:idx + len(v)]
        idx += len(v)
    N = len(all_vals)
    _, counts = np.unique(all_vals, return_counts=True)
    T = np.sum(counts**3 - counts) / (N - 1) if N > 1 else 0
    sigma2 = (N * (N + 1) - T) / 12
    pvals = {}
    for a, b in combinations(list(groups.keys()), 2):
        ra, rb = rank_groups[a], rank_groups[b]
        z = abs(ra.mean() - rb.mean()) / np.sqrt(sigma2 * (1/len(ra) + 1/len(rb)))
        pvals[(a, b)] = 2 * (1 - stats.norm.cdf(abs(z)))
    m = len(pvals)
    return {k: min(v * m, 1.0) for k, v in pvals.items()}

def sig_str(p):
    return ('***' if p < 0.001
            else '**' if p < 0.01
            else '*' if p < 0.05
            else 'ns')

# =============================================================================
# 3. Colors and short y-axis labels
# =============================================================================
CLUSTER_PT   = {1: '#A93226', 2: '#3498DB', 3: '#1F3A5F'}
CLUSTER_FILL = {1: '#E8B4B0', 2: '#9CC4E0', 3: '#7891B5'}
CLUSTER_EDGE = {1: '#C0392B', 2: '#5DADE2', 3: '#2C3E7A'}

short_names_box = {
    'POP density (pop/km2)':      'Pop. Density\n(pop/km²)',
    'GDP per capita (thousands)': 'GDP per capita\n(thousand USD)',
    'GDP_Primary %':              'GDP Primary %',
    'GDP_Secondary %':            'GDP Secondary %',
    'Female Population%':         'Female Pop. %',
    'Under 15 years old%':        'Under-15 Pop. %',
    'Over 65 years old%':         'Over-65 Pop. %',
    'VOTER_TURNOUT_ratio':        'Voter Turnout %',
    'Nonprofit per 10k pop':      'Nonprofit Orgs.\nper 10k pop',
    'AVG_ANNUAL_WAGE':            'Avg. Gov. Annual\nWage (thousand USD)',
    'EXP_GDP_RATIO':              'Direct Exp. / GDP %',
    'establishment entry rate':   'Estab. Entry Rate %',
    'Fed Cor Conv per 100k':      'Fed. Corruption\nper 100k',
}

# =============================================================================
# 4. Boxplot grid (auto n_rows × 3)
# =============================================================================
n_factors = len(factor_cols)
n_cols = 3
n_rows = math.ceil(n_factors / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4.2))
axes_flat = axes.flatten() if n_rows > 1 else np.array([axes]).flatten()

kw_rows = []

for idx, col in enumerate(factor_cols):
    ax = axes_flat[idx]
    vals = df[col].values
    groups = {c: vals[labels == c] for c in sorted(set(labels))}
    H, p_kw = stats.kruskal(*[groups[c] for c in sorted(groups.keys())])
    dunn = dunn_test(groups) if p_kw < 0.05 else {(a, b): np.nan for a, b in combinations(groups.keys(), 2)}
    cl = sorted(groups.keys())

    bp = ax.boxplot([groups[c] for c in cl],
                    positions=list(range(1, len(cl) + 1)),
                    widths=0.55, patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', linewidth=1.8),
                    whiskerprops=dict(color='#444', linewidth=1.2),
                    capprops=dict(color='#444', linewidth=1.2),
                    boxprops=dict(linewidth=1.2))
    for patch, c in zip(bp['boxes'], cl):
        patch.set_facecolor(CLUSTER_FILL[c])
        patch.set_edgecolor(CLUSTER_EDGE[c])

    # Scatter with jitter
    rng_l = np.random.default_rng(0)
    for i, c in enumerate(cl):
        v = groups[c]
        ax.scatter(i + 1 + (rng_l.random(len(v)) - 0.5) * 0.28, v,
                   s=16, color=CLUSTER_PT[c], alpha=0.55,
                   edgecolor='white', linewidth=0.4, zorder=3)

    ax.set_xticks(list(range(1, len(cl) + 1)))
    ax.set_xticklabels([f'Typology {c}' for c in cl], fontsize=14)
    ax.set_ylabel(short_names_box.get(col, col), fontsize=14, fontweight='bold')
    ax.tick_params(axis='both', labelsize=14)
    ax.grid(True, axis='y', alpha=0.25, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Significance brackets
    ymax = max(np.max(groups[c]) for c in cl)
    ymin = min(np.min(groups[c]) for c in cl)
    ry = ymax - ymin
    y_top = ymax + 0.07 * ry
    step = 0.10 * ry
    pairs = [((cl[0], cl[1]), 0), ((cl[1], cl[2]), 1), ((cl[0], cl[2]), 2)]
    for (a, b), level in pairs:
        yb = y_top + step * level
        ax.plot([cl.index(a) + 1, cl.index(a) + 1,
                 cl.index(b) + 1, cl.index(b) + 1],
                [yb, yb + 0.012*ry, yb + 0.012*ry, yb],
                color='black', lw=1.0)
        ax.text((cl.index(a) + cl.index(b) + 2) / 2,
                yb + 0.018*ry, sig_str(dunn[(a, b)]),
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_ylim(ymin - 0.05*ry, y_top + step*2 + 0.06*ry)

    # Save stats
    for c in cl:
        v = groups[c]
        kw_rows.append({
            'Factor': col, 'Typology': c, 'n': len(v),
            'median': round(np.median(v), 4),
            'mean': round(float(v.mean()), 4),
            'KW_H': round(H, 4), 'KW_p': p_kw,
            'Dunn_C1vC2_p': round(dunn[(cl[0], cl[1])], 4),
            'Dunn_C1vC3_p': round(dunn[(cl[0], cl[2])], 4),
            'Dunn_C2vC3_p': round(dunn[(cl[1], cl[2])], 4),
        })

for i in range(n_factors, len(axes_flat)):
    axes_flat[i].axis('off')

plt.tight_layout(h_pad=2.5, w_pad=2.0)

# =============================================================================
# 5. Export
# =============================================================================
fig.savefig(os.path.join(RESULTS_DIR, 'SDG_cluster_boxplots.png'),
            dpi=300, bbox_inches='tight', facecolor='white')

pd.DataFrame(kw_rows).to_excel(
    os.path.join(RESULTS_DIR, 'KW_Dunn_results.xlsx'), index=False)

plt.show()
print("\nDone. Figure and KW+Dunn results exported.")

In [ ]:
"""
SDG Synergy and Trade-off Analysis by Typology
"""
# =============================================================================
# Global font settings — Arial
# =============================================================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12

# =============================================================================
# 1. Load data
# =============================================================================
PATH_DATA = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result3_percentage and correlation.xlsx')

df = pd.read_excel(PATH_DATA)
# The first column has no header — rename it "Type"
df.columns = ['Type', 'Group', 'Ratio', 'Average']
df['Group'] = df['Group'].str.strip()
df['Type'] = df['Type'].str.strip()

# =============================================================================
# 2. Colors
# =============================================================================
BAR_COLORS = {
    'Overall':  '#B0B0B0',   # light gray
    'Cluster1': '#E8B4B0',   # light red
    'Cluster2': '#9CC4E0',   # light blue
    'Cluster3': '#7891B5',   # medium blue-gray
}
DOT_COLORS = {
    'Overall':  '#636363',   # dark gray
    'Cluster1': '#A93226',   # dark red
    'Cluster2': '#3498DB',   # bright blue
    'Cluster3': '#1F3A5F',   # dark navy
}

groups = ['Overall', 'Cluster1', 'Cluster2', 'Cluster3']
group_labels = ['All', 'Typology 1', 'Typology 2', 'Typology 3']

syn = df[df['Type'] == 'Synergy'].set_index('Group').reindex(groups)
trd = df[df['Type'] == 'Tradeoff'].set_index('Group').reindex(groups)

# =============================================================================
# 3. Plot
# =============================================================================
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

bar_width = 0.17
syn_center = 1.0
trd_center = 1.9
offsets = np.array([-1.5, -0.5, 0.5, 1.5]) * bar_width

for i, g in enumerate(groups):
    ax1.bar(syn_center + offsets[i], syn.loc[g, 'Ratio'], bar_width * 0.92,
            color=BAR_COLORS[g], edgecolor='none', zorder=2)
    ax1.bar(trd_center + offsets[i], trd.loc[g, 'Ratio'], bar_width * 0.92,
            color=BAR_COLORS[g], edgecolor='none', zorder=2)

for i, g in enumerate(groups):
    ax2.scatter(syn_center + offsets[i], syn.loc[g, 'Average'],
                s=70, color=DOT_COLORS[g], zorder=4, clip_on=False)
    ax2.scatter(trd_center + offsets[i], trd.loc[g, 'Average'],
                s=70, color=DOT_COLORS[g], zorder=4, clip_on=False)

# ---- Align 0 lines: both axes symmetric around 0 ----
ax1.set_ylim(-50, 50)
ax2.set_ylim(-0.75, 0.75)

ax1.set_ylabel('Percentage of significant correlations (%)', fontsize=14)
ax2.set_ylabel('Average correlation coefficient', fontsize=14)

# X-axis
ax1.set_xticks([syn_center, trd_center])
ax1.set_xticklabels(['Synergy', 'Trade-off'], fontsize=14)
ax1.tick_params(axis='y', labelsize=12)
ax2.tick_params(axis='y', labelsize=12)

# Remove horizontal grid lines
ax1.set_axisbelow(True)
ax1.grid(False)
ax2.grid(False)

# Horizontal 0-line (both axes 0 overlap at this level)
ax1.axhline(0, color='#AAAAAA', lw=0.7, zorder=1)

# Vertical separator between Synergy and Trade-off groups
mid_x = (syn_center + trd_center) / 2
ax1.axvline(mid_x, color='#CCCCCC', lw=0.7, ls='--', zorder=0)

ax1.set_xlim(syn_center - 0.50, trd_center + 0.50)
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)

# =============================================================================
# 4. Custom legend (bar + dot combined for each group)
# =============================================================================
class BarDotHandler(HandlerBase):
    def __init__(self, bar_color, dot_color):
        self.bar_color = bar_color
        self.dot_color = dot_color
        super().__init__()
    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        rect = FancyBboxPatch((xdescent, ydescent), width, height,
                              boxstyle="square,pad=0",
                              facecolor=self.bar_color, edgecolor='none',
                              alpha=0.6, transform=trans)
        dot = mlines.Line2D([xdescent + width/2], [ydescent + height/2],
                            marker='o', markersize=6, color=self.dot_color,
                            markeredgecolor=self.dot_color, linestyle='None',
                            transform=trans)
        return [rect, dot]

legend_items = [mpatches.Patch(facecolor=BAR_COLORS[g]) for g in groups]
handler_map = {legend_items[i]: BarDotHandler(BAR_COLORS[g], DOT_COLORS[g])
               for i, g in enumerate(groups)}
ax1.legend(legend_items, group_labels, handler_map=handler_map,
           loc='upper right', fontsize=12, frameon=True, framealpha=0.95,
           edgecolor='#cccccc', title='Group', title_fontsize=12)

plt.tight_layout()

# =============================================================================
# 5. Export
# =============================================================================
fig.savefig(os.path.join(RESULTS_DIR, 'SDG_synergy_tradeoff.png'),
            dpi=200, bbox_inches='tight', facecolor='white')

plt.show()
print("Done. Figure saved as SDG_synergy_tradeoff.png")

In [ ]:
"""
SDG Pair Dominance Triangle Heatmap
Border/fill rules:
  - No correlation (all 3 == 0):            gray cell
  - Consistent dominance (all >=0 or all <=0, not all 0): dashed border
  - Shifting dominance (mixed signs):       solid border
"""
# =============================================================================
# Global font settings
# =============================================================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12

# =============================================================================
# 1. Load data
# =============================================================================
PATH_DATA = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result3_intensity_Dominance Score_S-T.xlsx')

df = pd.read_excel(PATH_DATA)
df.columns = [c.strip() for c in df.columns]
print(df.head())

# =============================================================================
# 2. Classify each pair
# =============================================================================
def classify(row):
    d = np.array([row['dominance1'], row['dominance2'], row['dominance3']])
    if np.all(d == 0):
        return 'no_corr'
    if np.all(d > 0) or np.all(d < 0):
        return 'consistent'   
    return 'shifting'

df['cls'] = df.apply(classify, axis=1)

# =============================================================================
# 3. Build the SDG × SDG lower-triangle matrix
# =============================================================================
def sdg_num(s):
    m = re.match(r'SDG(\d+)', s)
    return int(m.group(1)) if m else -1

sdg_labels = sorted(set(df['Object1']).union(df['Object2']), key=sdg_num)
n = len(sdg_labels)
sdg_to_i = {s: i for i, s in enumerate(sdg_labels)}
print(f"SDGs ({n}):", sdg_labels)

# For each cell (row r, col c) we store 3 values (d1, d2, d3) and a class
val_mat = np.full((n, n, 3), np.nan)
cls_mat = np.full((n, n), '', dtype=object)

for _, row in df.iterrows():
    i, j = sdg_to_i[row['Object1']], sdg_to_i[row['Object2']]
    r, c = min(i, j), max(i, j)   
    val_mat[r, c] = [row['dominance1'], row['dominance2'], row['dominance3']]
    cls_mat[r, c] = row['cls']

# =============================================================================
# 4. Colormap: red (tradeoff) ← white ← blue (synergy)
# =============================================================================
cmap = LinearSegmentedColormap.from_list('dominance',
    ['#A93226', '#E8B4B0', '#FFFFFF', '#7891B5', '#1F3A5F'], N=256)

all_vals = val_mat[~np.isnan(val_mat)]
vmin = float(all_vals.min())     # -0.337
vmax = float(all_vals.max())     #  0.453

norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
# =============================================================================
# 5. Plot
# =============================================================================
fig, ax = plt.subplots(figsize=(15, 15))

CELL_SIZE = 1.0
GAP = 0.14              
inner = CELL_SIZE - GAP 
half = inner / 2
strip_w = inner / 3     

for r in range(n):
    for c in range(n):
        if c < r:   
            continue

        cls = cls_mat[r, c]
        vals = val_mat[r, c]

        cx = c + 0.5
        # Flip y so SDG1 is on top-left
        cy = (n - 1 - r) + 0.5

        # Bottom-left corner of the framed area
        x0 = cx - half
        y0 = cy - half

        if cls == 'no_corr':
            # Gray cell, solid border
            ax.add_patch(Rectangle((x0, y0), inner, inner,
                                   facecolor='#888888', edgecolor='black',
                                   linewidth=1.2, linestyle='-'))
            continue

        # 3 vertical stripes for cluster 1, 2, 3
        for k in range(3):
            stripe_color = cmap(norm(vals[k]))
            ax.add_patch(Rectangle((x0 + k * strip_w, y0),
                                   strip_w, inner,
                                   facecolor=stripe_color,
                                   edgecolor='none', linewidth=0))

        # Outer border on top of stripes (dashed for consistent, solid+thick for shifting)
        if cls == 'consistent':
            linestyle = '--'
            lw = 1.4
        else:  
            linestyle = '-'
            lw = 2.5   

        ax.add_patch(Rectangle((x0, y0), inner, inner,
                               facecolor='none', edgecolor='black',
                               linewidth=lw, linestyle=linestyle))

# Axis limits
ax.set_xlim(-0.3, n + 0.1)
ax.set_ylim(-2.5, n + 0.1)
ax.set_aspect('equal')

# X labels on top
ax.set_xticks(np.arange(n) + 0.5)
ax.set_xticklabels(sdg_labels, fontsize=12, rotation=90)
ax.xaxis.tick_top()

# Y labels on right
ax.set_yticks(np.arange(n) + 0.5)
ax.set_yticklabels(sdg_labels[::-1], fontsize=12)
ax.yaxis.tick_right()

# Hide spines
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis='both', length=0)

# =============================================================================
# 6. Legend (dominance type) — bottom-left area
# =============================================================================
lx = 0.3
ly = 3.5
gap_y = 1.3

ax.text(lx, ly + 2.3, 'Dominance type',
        fontsize=14, fontweight='bold', va='center')

# Consistent (dashed)
ax.add_patch(Rectangle((lx, ly + 1.0), 1.0, 0.8,
                       facecolor='white', edgecolor='black',
                       linewidth=1.4, linestyle='--'))
ax.text(lx + 1.35, ly + 1.4, 'Consistent dominance',
        fontsize=12, va='center')

# Shifting (solid)
ax.add_patch(Rectangle((lx, ly - 0.2), 1.0, 0.8,
                       facecolor='white', edgecolor='black',
                       linewidth=2.5, linestyle='-'))
ax.text(lx + 1.35, ly + 0.2, 'Shifting dominance',
        fontsize=12, va='center')

# No correlation (gray)
ax.add_patch(Rectangle((lx, ly - 1.4), 1.0, 0.8,
                       facecolor='#888888', edgecolor='black', linewidth=1.2))
ax.text(lx + 1.35, ly - 1.0, 'No correlation',
        fontsize=12, va='center')

# =============================================================================
# 7. Colorbar (aligned with "Dominance score" title, matching legend spacing)
# =============================================================================
ax.text(lx, 1.4, 'Dominance score',
        fontsize=14, fontweight='bold', va='center')
 
cbar_ax = fig.add_axes([0.202, 0.24, 0.25, 0.018])
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax,
                  orientation='horizontal')
cb.set_ticks([vmin, 0, vmax])
cb.set_ticklabels([f'{vmin:.3f}', '0', f'{vmax:.3f}'])
cb.ax.tick_params(labelsize=12)
cb.outline.set_visible(False)
 
fig.text(0.202, 0.21, 'Tradeoff dominance', fontsize=12, ha='center')
fig.text(0.325, 0.21, 'No dominance', fontsize=12, ha='center')
fig.text(0.452, 0.21, 'Synergy dominance', fontsize=12, ha='center')
 

# =============================================================================
# 8. Export
# =============================================================================
fig.savefig(os.path.join(RESULTS_DIR, 'SDG_dominance_triangle.png'),
            dpi=200, bbox_inches='tight', facecolor='white')

# Also export the classification
out = df[['Object1', 'Object2', 'dominance1', 'dominance2', 'dominance3', 'cls']].copy()
out.columns = ['SDG_A', 'SDG_B', 'Typology1', 'Typology2', 'Typology3', 'dominance_type']
out.to_excel(os.path.join(RESULTS_DIR, 'dominance_classification.xlsx'), index=False)

plt.show()
print("\nDone.")
print(f"  Consistent dominance: {(df['cls']=='consistent').sum()}")
print(f"  Shifting dominance:   {(df['cls']=='shifting').sum()}")
print(f"  No correlation:       {(df['cls']=='no_corr').sum()}")

In [ ]:
"""
Rose/Radar Chart: Hub Scores for SDG Synergy and Trade-off across 4 Levels
"""
# =============================================================================
# Global font settings
# =============================================================================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['xtick.labelsize'] = 18
plt.rcParams['ytick.labelsize'] = 18
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['legend.fontsize'] = 18

# =============================================================================
# 1. Load data
# =============================================================================
PATH_DATA = os.path.join(DATA_DIR, 'Data_for_figures_4results', 'Result4_HubScore_Synergy and Tradeoff.xlsx')

df = pd.read_excel(PATH_DATA)
df.columns = [c.strip() for c in df.columns]

# =============================================================================
# 2. SDG ordering
# =============================================================================
def sdg_num(s):
    m = re.match(r'SDG(\d+)', s)
    return int(m.group(1)) if m else -1

sdg_labels = sorted(df['Object'].unique(), key=sdg_num)
n_sdg = len(sdg_labels)

angles = np.linspace(0, 2 * np.pi, n_sdg, endpoint=False).tolist()
angles += angles[:1]

# =============================================================================
# 3. Colors
# =============================================================================
SYN_COLOR = '#1F3A5F'   # dark navy
TRD_COLOR = '#A93226'   # dark red

# =============================================================================
# 4. Plot 1 × 4 grid
# =============================================================================
levels = ['Overall', 'Cluster1', 'Cluster2', 'Cluster3']
panel_labels = ['a.', 'b.', 'c.', 'd.']

fig, axes = plt.subplots(1, 4, figsize=(22, 6),
                         subplot_kw=dict(projection='polar'))

for idx, level in enumerate(levels):
    ax = axes[idx]
    sub = df[df['Level'] == level].set_index('Object').reindex(sdg_labels)

    syn_vals = sub['Synergy'].tolist() + [sub['Synergy'].tolist()[0]]
    trd_vals = sub['Tradeoff'].tolist() + [sub['Tradeoff'].tolist()[0]]

    # Synergy
    ax.plot(angles, syn_vals, color=SYN_COLOR, linewidth=2.8,
            marker='o', markersize=5,
            markerfacecolor='white', markeredgecolor=SYN_COLOR,
            markeredgewidth=1.2, zorder=3)
    ax.fill(angles, syn_vals, color=SYN_COLOR, alpha=0.15, zorder=2)

    # Trade-off
    ax.plot(angles, trd_vals, color=TRD_COLOR, linewidth=2.8,
            marker='o', markersize=5,
            markerfacecolor='white', markeredgecolor=TRD_COLOR,
            markeredgewidth=1.2, zorder=3)
    ax.fill(angles, trd_vals, color=TRD_COLOR, alpha=0.15, zorder=2)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(sdg_labels, fontsize=14)

    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels([])
    ax.tick_params(axis='x', pad=15)

    for r_val in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.text(0, r_val, f'{r_val:.1f}',
                fontsize=12, color='#666666',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                          edgecolor='none', alpha=0),
                zorder=4)

    ax.grid(True, alpha=0.55, linewidth=0.9, dashes=(4, 1.5))
    ax.spines['polar'].set_edgecolor('#333333')
    ax.spines['polar'].set_linewidth(1.0)

    ax.text(-0.12, 1.10, panel_labels[idx],
            transform=ax.transAxes, fontsize=22, fontweight='bold',
            ha='left', va='top')
    title_map = {'Overall': 'Overall', 'Cluster1': 'Typology 1',
                 'Cluster2': 'Typology 2', 'Cluster3': 'Typology 3'}
    ax.set_title(title_map[level], fontsize=20, pad=25)

# =============================================================================
# 5. Shared bottom legend
# =============================================================================
legend_elements = [
    Line2D([0], [0], color=SYN_COLOR, lw=2.8, marker='o',
           markerfacecolor='white', markeredgecolor=SYN_COLOR,
           markeredgewidth=1.4, markersize=5, label='Synergy'),
    Line2D([0], [0], color=TRD_COLOR, lw=2.8, marker='o',
           markerfacecolor='white', markeredgecolor=TRD_COLOR,
           markeredgewidth=1.4, markersize=5, label='Trade-off'),
]
fig.legend(handles=legend_elements, loc='lower center',
           bbox_to_anchor=(0.5, -0.02),
           ncol=2, fontsize=18, frameon=False,
           handlelength=2.5, handletextpad=0.8, columnspacing=3.0)

plt.subplots_adjust(left=0.03, right=0.98, top=0.90, bottom=0.10, wspace=0.35)

# =============================================================================
# 6. Export
# =============================================================================
fig.savefig(os.path.join(RESULTS_DIR, 'SDG_hub_scores_radar.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Done.")